# Transformacja WLASL do uniwersalnego CSV

Ten notebook transformuje pierwszy zbior danych (WLASL) do jednego formatu CSV.
Wynik jest zapisywany do osobnego katalogu merge: merged_datasets/universal_metadata.csv.

Docelowe kolumny:
- label
- source
- video_path
- start_frame
- end_frame
- length_frames
- fps
- signer_id
- has_video
- video_width
- video_height

In [ ]:
from pathlib import Path
import csv
import json
import cv2

# Ustawienia sciezek
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
WLASL_JSON = PROJECT_ROOT / 'kaggle_dataset' / 'WLASL_v0.3.json'
VIDEOS_DIR = PROJECT_ROOT / 'kaggle_dataset' / 'videos'
MERGED_DIR = PROJECT_ROOT / 'merged_datasets'
OUTPUT_CSV = MERGED_DIR / 'universal_metadata.csv'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('WLASL_JSON exists:', WLASL_JSON.exists())
print('VIDEOS_DIR exists:', VIDEOS_DIR.exists())
print('MERGED_DIR:', MERGED_DIR)
print('OUTPUT_CSV:', OUTPUT_CSV)

In [ ]:
def probe_video(video_path: Path):
    """Zwraca (total_frames, fps, width, height) albo None-y, jesli nie da sie odczytac."""
    if not video_path.exists():
        return None, None, None, None

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None, None, None, None

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    if total_frames <= 0:
        total_frames = None
    if fps <= 0:
        fps = None
    if width <= 0:
        width = None
    if height <= 0:
        height = None

    return total_frames, fps, width, height


with WLASL_JSON.open('r', encoding='utf-8') as f:
    data = json.load(f)

total = sum(len(item.get('instances', [])) for item in data)
PRINT_EVERY = 200
processed = 0

rows = []
for item in data:
    label = item.get('gloss')

    for inst in item.get('instances', []):
        processed += 1
        if processed == 1 or processed % PRINT_EVERY == 0 or processed == total:
            print(f'Przetworzono {processed}/{total}')

        video_id = str(inst.get('video_id', '')).strip()
        video_path = VIDEOS_DIR / f'{video_id}.mp4'

        has_video = video_path.exists()
        total_frames, video_fps, video_width, video_height = probe_video(video_path)

        start_frame = inst.get('frame_start')
        raw_end = inst.get('frame_end')

        try:
            start_frame = int(start_frame)
        except (TypeError, ValueError):
            start_frame = None

        try:
            raw_end = int(raw_end)
        except (TypeError, ValueError):
            raw_end = None

        if raw_end == -1:
            end_frame = total_frames
        else:
            end_frame = raw_end

        if start_frame is not None and end_frame is not None and end_frame >= start_frame:
            length_frames = end_frame - start_frame + 1
        else:
            length_frames = None

        fps = video_fps if video_fps is not None else inst.get('fps')
        if fps is not None and fps > 0 and length_frames is not None:
            duration_sec = round(length_frames / fps, 3)
        else:
            duration_sec = None

        # Sciezka wzgledna do katalogu repozytorium (szum/)
        try:
            rel_path = str(video_path.relative_to(PROJECT_ROOT)).replace('\\', '/')
        except ValueError:
            rel_path = str(video_path).replace('\\', '/')

        rows.append({
            'label': label,
            'source': inst.get('source'),
            'video_path': rel_path,
            'start_frame': start_frame,
            'end_frame': end_frame,
            'length_frames': length_frames,
            'duration_sec': duration_sec,
            'fps': fps,
            'signer_id': inst.get('signer_id'),
            'has_video': has_video,
            'video_width': video_width,
            'video_height': video_height,
        })

print('Liczba rekordow do zapisu:', len(rows))

In [ ]:
columns = [
    'label',
    'source',
    'video_path',
    'start_frame',
    'end_frame',
    'length_frames',
    'duration_sec',
    'fps',
    'signer_id',
    'has_video',
    'video_width',
    'video_height',
]

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_CSV.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    writer.writerows(rows)

print('Zapisano CSV:', OUTPUT_CSV)

In [ ]:
# Szybki podglad
for row in rows[:3]:
    print(row)